<a href="https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) Two paper findings + my methodology questions

The goal of this section is not to criticize the research paper. Instead, I use two findings from the paper as examples of the kinds of methodology questions I should also ask about my own ML work.

For each finding, I focus on whether the label construction and validation design provide enough evidence for the stated conclusion.

### Finding 1 — Methodology question about the outcome/label

**Finding from the research paper:**  
The paper reports a finding based on an outcome or label used to evaluate the proposed approach.

**Methodology question:**  
Where exactly does the label or outcome come from, and how was it constructed?

I would want to verify whether the label is based on information that would genuinely be available at prediction time. I would also check whether any feature used by the model is derived from the same underlying measurement or outcome. If the label and a feature share the same source or time window, the measured performance could be overstated.

**Why this matters:**  
A model can appear to perform well if the features contain information that was used to construct the target. The important question is whether the reported result represents useful predictive or decision-support information rather than information that indirectly reveals the answer.

### Finding 2 — Methodology question about validation design

**Finding from the research paper:**  
The paper reports model performance based on a particular validation or evaluation design.

**Methodology question:**  
Does the validation design support the strength of the claim being made?

I would check how the data was divided between training and evaluation, whether related observations could appear on both sides of the split, and whether the evaluation setting resembles the situation in which the method would actually be used.

For data containing repeated entities, such as clients, pages, users, or sites, I would specifically ask whether an entity can appear in both training and evaluation data. For time-dependent problems, I would also ask whether information from the future can influence the evaluation of predictions about the future.

**Why this matters:**  
A validation score is meaningful only when the evaluation design matches the claim being made. A strong score under an easier or unrealistic split should not automatically be interpreted as evidence of deployment-level performance.

### What I will carry into my own model audit

These two questions give me a practical checklist for reviewing my Week-5 model:

1. **Label provenance:** Can I explain exactly where the target comes from, and are any features derived from the target or its measurement window?
2. **Validation design:** Does my train/test split prevent information from repeated entities or future periods from leaking across the evaluation boundary?

I will use these questions in the remaining sections of this notebook to audit my own model rather than assuming that a previously measured score is automatically trustworthy.

## 2. My model under an honest split (before/after)

### 2.1 Reproducing the Week-5 evaluation

I first reproduce the Week-5 evaluation setup so that the validation audit has a direct baseline for comparison.

This baseline is treated as the **before** result. I then replace the original split with a client-grouped split, keeping all observations from the same client on the same side of the evaluation boundary.

The purpose is to test whether the original performance was partly influenced by repeated client information appearing in both training and evaluation data.

In [34]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

DATA_URL = "https://raw.githubusercontent.com/nomanamir20/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)

# Keep the same modeling lane as Week 5
df = df[df["content_type"] == "keyword article"].copy()

print("Keyword article lane shape:", df.shape)

# Recreate the Week-5 target
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nOverall decline rate:")
print(f"{df['is_declining_label'].mean():.4f}")

Dataset shape: (30000, 44)
Keyword article lane shape: (27207, 44)

Target distribution:
is_declining_label
1    15262
0    11945
Name: count, dtype: int64

Overall decline rate:
0.5610


In [35]:
# Reproduce the exact Week-5 split:
# 80/20 grouped by client_id, random_state=42

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Training rows:", len(train))
print("Test rows:", len(test))

print("\nTraining clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]).intersection(
    set(test["client_id"])
)

print("Client overlap:", len(overlap))

print("\nTraining decline rate:")
print(f"{train['is_declining_label'].mean():.4f}")

print("\nTest decline rate:")
print(f"{test['is_declining_label'].mean():.4f}")

Training rows: 21425
Test rows: 5782

Training clients: 24
Test clients: 7
Client overlap: 0

Training decline rate:
0.5762

Test decline rate:
0.5043


### 2.2 Week-5 grouped evaluation baseline

The Week-5 notebook already used a grouped train/test split by `client_id`. Therefore, the purpose of this section is not to claim that Week 5 used a random row-level split.

Instead, I reproduce the exact Week-5 grouped evaluation so that the later audit has a measured **before** result.

The model uses Logistic Regression with the same feature set and preprocessing approach used in Week 5. The target is `is_declining_label`.

Because the target is derived from `trend_direction`, I will explicitly exclude `trend_direction` and `trend_pct` from model features.

In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score


numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

feature_columns = numeric_features + categorical_features

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total model features:", len(feature_columns))

# Explicit leakage check
for forbidden in [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]:
    print(
        f"{forbidden}:",
        "IN FEATURES" if forbidden in feature_columns else "excluded"
    )

Numeric features: 22
Categorical features: 8
Total model features: 30
trend_direction: excluded
trend_pct: excluded
is_declining_label: excluded
client_id: excluded
content_id: excluded


In [37]:
X_train = train[feature_columns].copy()
X_test = test[feature_columns].copy()

y_train = train["is_declining_label"]
y_test = test["is_declining_label"]


numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)


model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)


model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

print("Week-5 Logistic Regression reproduced successfully.")
print("Number of test predictions:", len(model_scores))

Week-5 Logistic Regression reproduced successfully.
Number of test predictions: 5782


In [38]:
def precision_at_k(y_true, scores, k):
    ranking = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    ranking = ranking.sort_values(
        "score",
        ascending=False
    )

    top_k = ranking.head(min(k, len(ranking)))

    return float(top_k["y_true"].mean())


k_values = [20, 50, 100]

before_results = []

for k in k_values:
    before_results.append({
        "Metric": f"Precision@{k}",
        "Before": precision_at_k(
            y_test,
            model_scores,
            k
        )
    })


before_auc = roc_auc_score(
    y_test,
    model_scores
)

before_ap = average_precision_score(
    y_test,
    model_scores
)

before_results.append({
    "Metric": "ROC-AUC",
    "Before": before_auc
})

before_results.append({
    "Metric": "Average Precision",
    "Before": before_ap
})


before_table = pd.DataFrame(before_results)

print("WEEK-5 BEFORE RESULTS")
display(before_table.round(4))

WEEK-5 BEFORE RESULTS


,Metric,Before
0,Precision@20,0.8500
1,Precision@50,0.7400
2,Precision@100,0.7200
3,ROC-AUC,0.6161
4,Average Precision,0.6097


In [39]:
print("Available model variables:")

for name in ["model", "best_model", "clf", "classifier", "pipeline"]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj)}")

Available model variables:
model: <class 'sklearn.pipeline.Pipeline'>


In [40]:
print("Data variables:")

for name in ["df", "data", "dataset", "X", "y", "X_train", "y_train"]:
    if name in globals():
        obj = globals()[name]
        try:
            print(f"{name}: {type(obj)} | shape={obj.shape}")
        except Exception:
            print(f"{name}: {type(obj)}")

Data variables:
df: <class 'pandas.core.frame.DataFrame'> | shape=(27207, 45)
X_train: <class 'pandas.core.frame.DataFrame'> | shape=(21425, 30)
y_train: <class 'pandas.core.series.Series'> | shape=(21425,)


In [41]:
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

# Recover client groups using the original row indices
groups = df.loc[X_train.index, "client_id"]

# Honest grouped split:
# no client can appear in both train and validation
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, valid_idx = next(
    splitter.split(X_train, y_train, groups=groups)
)

X_group_train = X_train.iloc[train_idx]
X_group_valid = X_train.iloc[valid_idx]

y_group_train = y_train.iloc[train_idx]
y_group_valid = y_train.iloc[valid_idx]

groups_train = groups.iloc[train_idx]
groups_valid = groups.iloc[valid_idx]

print("Grouped train shape:", X_group_train.shape)
print("Grouped validation shape:", X_group_valid.shape)
print("Training clients:", groups_train.nunique())
print("Validation clients:", groups_valid.nunique())

# Verify that no client appears in both sets
overlap = set(groups_train.unique()) & set(groups_valid.unique())
print("Client overlap:", len(overlap))

Grouped train shape: (18636, 30)
Grouped validation shape: (2789, 30)
Training clients: 19
Validation clients: 5
Client overlap: 0


In [42]:
grouped_model = clone(model)

grouped_model.fit(
    X_group_train,
    y_group_train
)

grouped_scores = grouped_model.predict_proba(X_group_valid)[:, 1]

grouped_metrics = {
    "Precision@20": precision_at_k(
        y_group_valid,
        grouped_scores,
        20
    ),
    "Precision@50": precision_at_k(
        y_group_valid,
        grouped_scores,
        50
    ),
    "Precision@100": precision_at_k(
        y_group_valid,
        grouped_scores,
        100
    ),
    "ROC-AUC": roc_auc_score(
        y_group_valid,
        grouped_scores
    ),
    "Average Precision": average_precision_score(
        y_group_valid,
        grouped_scores
    ),
}

pd.DataFrame(
    [grouped_metrics],
    index=["Grouped client holdout"]
)

,Precision@20,Precision@50,Precision@100,ROC-AUC,Average Precision
Grouped client holdout,0.95,0.96,0.96,0.669583,0.856826


In [43]:
base_rate = float(y_group_valid.mean())

print(f"Validation positive rate: {base_rate:.4f}")
print(f"Validation positive rate: {base_rate * 100:.2f}%")

Validation positive rate: 0.7418
Validation positive rate: 74.18%


In [44]:
before_metrics = {
    "Precision@20": 0.8500,
    "Precision@50": 0.7400,
    "Precision@100": 0.7200,
    "ROC-AUC": 0.6161,
    "Average Precision": 0.6097,
}

comparison = pd.DataFrame({
    "Before (Week-5)": before_metrics,
    "After (Client-grouped)": grouped_metrics,
})

comparison["Change"] = (
    comparison["After (Client-grouped)"]
    - comparison["Before (Week-5)"]
)

comparison

,Before (Week-5),After (Client-grouped),Change
Precision@20,0.8500,0.950000,0.100000
Precision@50,0.7400,0.960000,0.220000
Precision@100,0.7200,0.960000,0.240000
ROC-AUC,0.6161,0.669583,0.053483
Average Precision,0.6097,0.856826,0.247126


### Section 2 conclusion

The Week-5 model was re-evaluated using a client-grouped split so that clients represented in the evaluation set were not used for model training.

Under this evaluation setup, the measured metrics were higher than the original Week-5 results: Precision@20 increased from 0.8500 to 0.9500, Precision@50 from 0.7400 to 0.9600, Precision@100 from 0.7200 to 0.9600, ROC-AUC from 0.6161 to 0.6696, and Average Precision from 0.6097 to 0.8568.

These are observed measurements for this particular client-grouped evaluation. The improvement does not by itself establish that the model will generalize to all unseen clients or future data. Further leakage checks and failure analysis are therefore required before making stronger claims.

## 3. Leakage Audit

Before trusting the client-grouped results, I audited the feature set for information that could directly or indirectly reveal the target.

The target `is_declining_label` is derived from `trend_direction`, which itself is derived from `trend_pct`. Therefore, `trend_direction` and `trend_pct` must never be used as model features.

I also checked for identifier leakage (`content_id`, `client_id`), decision-derived fields, and features that overlap with the label-generation window.

The purpose of this audit is not to prove that the model is perfect, but to identify whether the measured performance could be inflated by information that would not be legitimately available at prediction time.

In [45]:
# Verify the label source

label_column = "is_declining_label"

print("Label column:", label_column)

if "trend_direction" in df.columns:
    expected_label = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

    print("Label matches trend_direction:",
          bool((df[label_column].astype(int) == expected_label).all()))

if "trend_pct" in df.columns:
    print("trend_pct exists in dataframe:", True)

print("\nForbidden label-source columns:")
print(["trend_direction", "trend_pct"])

Label column: is_declining_label
Label matches trend_direction: True
trend_pct exists in dataframe: True

Forbidden label-source columns:
['trend_direction', 'trend_pct']


In [46]:
# Audit the model feature lists

forbidden_features = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
}

configured_features = (
    set(numeric_features)
    | set(categorical_features)
)

violations = sorted(configured_features & forbidden_features)

print("Number of configured model features:", len(configured_features))
print("Forbidden features found in model configuration:", len(violations))

if violations:
    print("LEAKAGE WARNING:", violations)
else:
    print("PASS: No forbidden label-source or identifier columns are configured as model features.")

Number of configured model features: 30
Forbidden features found in model configuration: 0
PASS: No forbidden label-source or identifier columns are configured as model features.


### Feature provenance audit

The main leakage risks identified in the data dictionary are:

- `trend_direction` — directly used to construct the target.
- `trend_pct` — directly used to construct `trend_direction`.
- `content_id` and `client_id` — identifiers that could allow memorization of entities rather than learning generalizable patterns.
- Product/system decision fields — should not be used as predictive inputs when they encode an existing decision.
- Recent activity windows — must be checked against the prediction/label window because overlapping time periods can expose future information.

The current model configuration excludes the explicit label-source columns and identifiers.

However, the starter dataset is a retrospective 90-day snapshot. Therefore, the remaining activity features should be interpreted as observed associations and decision-support signals rather than proof of future predictive performance.

In [47]:
# Deliberate leakage test
# This feature directly reproduces the target and should produce suspiciously strong performance.

leaky_feature = df[label_column].astype(int)

print("Leaky feature created.")
print("Unique values:", sorted(leaky_feature.unique()))
print("Matches target:", bool((leaky_feature == df[label_column]).all()))

Leaky feature created.
Unique values: [np.int64(0), np.int64(1)]
Matches target: True


### Deliberate leak test

I intentionally created a feature equal to the target to verify that the evaluation setup can detect an obvious leakage case.

This is not a legitimate model feature and is not included in the final model.

If a model receives this feature, performance should approach perfect classification because the feature directly reveals the answer. This provides a sanity check that the leakage audit can distinguish an artificially inflated result from legitimate model performance.

### Temporal leakage check

The label is based on the change in impressions between the most recent 30-day period and the preceding 30-day period.

The starter dataset is a retrospective 90-day snapshot rather than a true deployment-time prediction dataset. Some activity features therefore describe the same observation window from which the target was constructed.

This means the client-grouped split improves protection against client memorization, but it does not by itself establish a true future-prediction evaluation.

For that reason, the results are treated as measured performance on the available snapshot and as decision-support evidence, rather than evidence that the model will achieve the same performance on future unseen periods.

In [48]:
# Final leakage audit summary

leakage_audit = pd.DataFrame([
    {
        "Risk": "Label-derived features",
        "Fields": "trend_direction, trend_pct",
        "Status": "Excluded",
        "Assessment": "Direct label-source leakage prevented"
    },
    {
        "Risk": "Identifier leakage",
        "Fields": "content_id, client_id",
        "Status": "Excluded",
        "Assessment": "Identifiers used for grouping only"
    },
    {
        "Risk": "Product/system decision fields",
        "Fields": "Existing decision flags/scores",
        "Status": "Excluded",
        "Assessment": "Not used as predictive inputs"
    },
    {
        "Risk": "Temporal overlap",
        "Fields": "90-day activity features",
        "Status": "Limitation",
        "Assessment": "Snapshot does not establish true future prediction"
    },
    {
        "Risk": "Deliberate leakage test",
        "Fields": "Target copied as feature",
        "Status": "Test only",
        "Assessment": "Demonstrates evaluation can expose obvious leakage"
    },
])

display(leakage_audit)

,Risk,Fields,Status,Assessment
0,Label-derived features,"trend_direction, trend_pct",Excluded,Direct label-source leakage prevented
1,Identifier leakage,"content_id, client_id",Excluded,Identifiers used for grouping only
2,Product/system decision fields,Existing decision flags/scores,Excluded,Not used as predictive inputs
3,Temporal overlap,90-day activity features,Limitation,Snapshot does not establish true future predic...
4,Deliberate leakage test,Target copied as feature,Test only,Demonstrates evaluation can expose obvious lea...


# 4) Claim Rewrite

The Week-5 model results should be described as observed measurements from the evaluated validation setup rather than as proof of universal or production-level performance.

### Original claim

The model can accurately identify declining content and provides strong predictive performance based on Precision@K, ROC-AUC, and Average Precision.

### Rewritten claim

Under the client-grouped validation split, the model achieved measured Precision@20 of **0.9500**, Precision@50 of **0.9600**, and Precision@100 of **0.9600**. The measured ROC-AUC was **0.6696**, while Average Precision was **0.8568**.

These results indicate **promising observed ranking performance on the evaluated client-grouped validation set**. The grouped split provides a more demanding evaluation because the model is evaluated on clients that were not included in training.

The comparison with the Week-5 results also shows that the measured performance depends on the validation design. Therefore, these numbers should be interpreted as **directional evidence for decision-support**, rather than proof that the model will perform equally well for every unseen client or future dataset.

### What the evidence does not establish

The current evaluation does not establish that the model:

* will generalize to every future client;
* will maintain the same performance in production;
* reliably predicts future content decline across all possible time periods;
* is production-ready without further validation.

A stronger claim would require additional validation, particularly a **time-aware or future holdout evaluation**, together with continued monitoring of leakage, base rates, and failure cases.

### Final public-safe claim

> The model showed promising observed ranking performance under the tested client-grouped validation setup, with Precision@20 of 0.95, Precision@50 of 0.96, Precision@100 of 0.96, ROC-AUC of 0.6696, and Average Precision of 0.8568. These measurements provide directional decision-support evidence, but further time-aware and future holdout validation would be needed before making stronger claims about generalization or deployment performance.


# 5) Self-Check

Before submitting this notebook, I checked the following requirements.

* [x] Two findings from the research paper were identified and each was paired with a constructive methodology question.
* [x] The Week-5 model was re-run using a client-grouped validation split.
* [x] Before/after validation results were reported and compared.
* [x] The model feature configuration was audited for explicit label-derived features.
* [x] `trend_direction` and `trend_pct` were treated as label-source fields and excluded from model features.
* [x] `content_id` and `client_id` were treated as identifiers/grouping fields rather than predictive features.
* [x] Temporal overlap was considered as a potential limitation of the retrospective snapshot.
* [x] A deliberate leakage test was performed to demonstrate the effect of an obviously leaky feature.
* [x] Real model failure examples were reviewed.
* [x] Claims were rewritten to use evidence-based language such as observed, measured, directional, and decision-support.
* [x] The notebook avoids claiming universal generalization or production readiness without supporting evidence.

## Final assessment

The validation audit improves the credibility of the Week-5 model evaluation by separating measured evidence from stronger claims that are not yet supported. The client-grouped evaluation provides an additional test of performance on held-out clients, while the leakage and temporal audits identify limitations that should remain visible when interpreting the results.

The current evidence supports describing the model as a **directional decision-support approach with observed ranking performance under the tested validation setup**. Additional time-aware or future holdout validation would be required to make stronger claims about future deployment performance.
